In [10]:
import os, json, math, time
import numpy as np
import pandas as pd

ROOT = "/kaggle/input/chexpert"

print("ROOT exists:", os.path.isdir(ROOT))
print("ROOT listing:", os.listdir(ROOT)[:20])

TRAIN_CSV = os.path.join(ROOT, "train.csv")
VALID_CSV = os.path.join(ROOT, "valid.csv")

# Some Kaggle versions store images in train/ valid/ at root; adapt if needed
TRAIN_DIR = os.path.join(ROOT, "train")
VALID_DIR = os.path.join(ROOT, "valid")

print("train.csv exists:", os.path.isfile(TRAIN_CSV))
print("valid.csv exists:", os.path.isfile(VALID_CSV))
print("train dir exists:", os.path.isdir(TRAIN_DIR))
print("valid dir exists:", os.path.isdir(VALID_DIR))


ROOT exists: True
ROOT listing: ['valid.csv', 'valid', 'train.csv', 'train']
train.csv exists: True
valid.csv exists: True
train dir exists: True
valid dir exists: True


In [11]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import densenet121, DenseNet121_Weights
from PIL import Image

from sklearn.metrics import roc_auc_score

LABELS = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
    'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
]

DISEASE_LABELS = [l for l in LABELS if l not in ("No Finding", "Support Devices")]


In [12]:
def resolve_path(rel_path: str) -> str:
    # CSV often has "CheXpert-v1.0-small/..." prefix
    rel_path = rel_path.replace("CheXpert-v1.0-small/", "")
    return os.path.join(ROOT, rel_path)

class CheXpertDataset(Dataset):
    def __init__(self, csv_path, img_size=224, only_frontal=False):
        self.df = pd.read_csv(csv_path)

        # Optional: keep only frontal views (recommended for first training)
        if only_frontal:
            self.df = self.df[self.df["Frontal/Lateral"] == "Frontal"].reset_index(drop=True)

        self.img_size = img_size
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = resolve_path(row["Path"])

        img = Image.open(img_path).convert("L")
        x = self.transform(img)

        y = np.zeros(len(LABELS), dtype=np.float32)
        m = np.zeros(len(LABELS), dtype=np.float32)  # mask: 1 if label is known 0/1, else 0

        for j, lab in enumerate(LABELS):
            v = row.get(lab, np.nan)
            if pd.isna(v):
                y[j] = 0.0
                m[j] = 0.0
            else:
                v = float(v)
                if v == -1.0:
                    y[j] = 0.0
                    m[j] = 0.0   # ignore uncertain
                else:
                    y[j] = 1.0 if v == 1.0 else 0.0
                    m[j] = 1.0

        return x, torch.from_numpy(y), torch.from_numpy(m)


In [13]:
IMG_SIZE = 224
BATCH_SIZE = 32   # on CPU, start modest
NUM_WORKERS = 2   # Kaggle CPU usually okay with 2–4

train_ds = CheXpertDataset(TRAIN_CSV, img_size=IMG_SIZE, only_frontal=True)
valid_ds = CheXpertDataset(VALID_CSV, img_size=IMG_SIZE, only_frontal=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=False)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)

print("Train size:", len(train_ds), "Valid size:", len(valid_ds))


Train size: 191027 Valid size: 202


In [15]:
def compute_pos_weight(dataset: CheXpertDataset):
    # Count positives/negatives where mask==1
    pos = np.zeros(len(LABELS), dtype=np.float64)
    neg = np.zeros(len(LABELS), dtype=np.float64)

    for i in range(len(dataset)):
        _, y, m = dataset[i]
        y = y.numpy()
        m = m.numpy()
        pos += (y == 1.0) * (m == 1.0)
        neg += (y == 0.0) * (m == 1.0)

    # pos_weight = neg/pos (avoid div by zero)
    pw = np.ones(len(LABELS), dtype=np.float32)
    for j in range(len(LABELS)):
        if pos[j] > 0:
            pw[j] = float(neg[j] / pos[j])
        else:
            pw[j] = 1.0
    return torch.tensor(pw, dtype=torch.float32)

pos_weight = compute_pos_weight(train_ds)
print("pos_weight (first few):", pos_weight[:6])


pos_weight (first few): tensor([0.0000, 1.6703, 0.3365, 0.0536, 0.1102, 0.3204])


In [18]:
def train_one_epoch(model, loader):
    model.train()
    running = 0.0
    n = 0
    for x, y, m in loader:
        x, y, m = x.to(device), y.to(device), m.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = masked_bce_loss(logits, y, m)
        loss.backward()
        optimizer.step()
        running += float(loss.item()) * x.size(0)
        n += x.size(0)
    return running / max(n, 1)

@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    all_logits, all_y, all_m = [], [], []
    for x, y, m in loader:
        x = x.to(device)
        logits = model(x).cpu()
        all_logits.append(logits)
        all_y.append(y)
        all_m.append(m)
    return torch.cat(all_logits), torch.cat(all_y), torch.cat(all_m)

def auroc_per_label(logits, y, m):
    probs = torch.sigmoid(logits).numpy()
    y = y.numpy()
    m = m.numpy()

    aucs = {}
    for j, lab in enumerate(LABELS):
        # compute only where mask==1 and both classes exist
        idx = m[:, j] == 1.0
        if idx.sum() < 20:
            aucs[lab] = None
            continue
        yj = y[idx, j]
        pj = probs[idx, j]
        if len(np.unique(yj)) < 2:
            aucs[lab] = None
            continue
        aucs[lab] = float(roc_auc_score(yj, pj))
    return aucs

EPOCHS = 2
for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    tr_loss = train_one_epoch(model, train_loader)
    logits, y, m = predict_all(model, valid_loader)
    aucs = auroc_per_label(logits, y, m)

    # print top few
    scored = [(k,v) for k,v in aucs.items() if v is not None]
    scored.sort(key=lambda x: x[1], reverse=True)
    print(f"Epoch {epoch}/{EPOCHS} | train_loss={tr_loss:.4f} | time={time.time()-t0:.1f}s")
    print("Top AUCs:", scored[:6])


NameError: name 'model' is not defined

In [ ]:
@torch.no_grad()
def collect_valid_arrays(model, valid_ds):
    loader = DataLoader(valid_ds, batch_size=64, shuffle=False, num_workers=2)
    logits, y, m = predict_all(model, loader)
    probs = torch.sigmoid(logits).numpy()
    return probs, y.numpy(), m.numpy(), valid_ds.df.copy()

def compute_thresholds(probs, y, m, df_valid, target_fp_rate=0.02):
    thresholds = {}

    # Identify "No Finding" positives in the CSV (ground truth)
    nf = (df_valid["No Finding"] == 1.0).values

    for j, lab in enumerate(LABELS):
        if lab in ("No Finding", "Support Devices"):
            continue

        # Scores on "No Finding" images
        p_nf = probs[nf, j]
        if p_nf.size > 0:
            # threshold at (1 - target_fp_rate) percentile
            thr_fp = float(np.quantile(p_nf, 1.0 - target_fp_rate))
        else:
            thr_fp = 0.5

        # Also compute F1-max threshold on known labels for this label
        idx = (m[:, j] == 1.0)
        yj = y[idx, j]
        pj = probs[idx, j]
        if idx.sum() < 50 or len(np.unique(yj)) < 2:
            thresholds[lab] = thr_fp
            continue

        best_thr, best_f1 = 0.5, -1.0
        for thr in np.linspace(0.05, 0.95, 91):
            pred = (pj >= thr).astype(np.int32)
            tp = int(((pred == 1) & (yj == 1)).sum())
            fp = int(((pred == 1) & (yj == 0)).sum())
            fn = int(((pred == 0) & (yj == 1)).sum())
            denom = (2*tp + fp + fn)
            f1 = (2*tp / denom) if denom > 0 else 0.0
            if f1 > best_f1:
                best_f1, best_thr = f1, float(thr)

        # Conservative: take the stricter (higher) of the two
        thresholds[lab] = float(max(thr_fp, best_thr))

    return thresholds

probs, yv, mv, dfv = collect_valid_arrays(model, valid_ds)
thresholds = compute_thresholds(probs, yv, mv, dfv, target_fp_rate=0.02)

print("Example thresholds:", {k: thresholds[k] for k in list(thresholds)[:5]})


In [ ]:
os.makedirs("/kaggle/working/models", exist_ok=True)

model_path = "/kaggle/working/models/chexpert_densenet121.pth"
thr_path   = "/kaggle/working/models/thresholds.json"
auc_path   = "/kaggle/working/models/valid_aucs.json"

# Save weights + label list
torch.save({
    "state_dict": model.state_dict(),
    "labels": LABELS,
    "img_size": IMG_SIZE
}, model_path)

with open(thr_path, "w") as f:
    json.dump(thresholds, f, indent=2)

# Save AUCs too
logits, y, m = predict_all(model, valid_loader)
aucs = auroc_per_label(logits, y, m)
with open(auc_path, "w") as f:
    json.dump(aucs, f, indent=2)

print("Saved:", model_path)
print("Saved:", thr_path)
print("Saved:", auc_path)
